# Medical LLM Fine-Tuning Pipeline

🧠 Problem

Medical terminology is complex and difficult for non-medical users to understand. Most existing explanations are:

Too technical
Not contextual
Not easily accessible in simple language

Additionally:

Public datasets are inconsistent (different formats)
Training large language models requires high compute
Managing multiple datasets and configurations is messy

💡 Solution Approach

This project solves the problem by:

Standardizing multiple datasets into a single format
Using prompt-based supervised fine-tuning
Leveraging LoRA (Low-Rank Adaptation) for efficient training
Using Unsloth for faster and memory-efficient LLM training
Building a modular pipeline:


Config → Dataset → Model → Training → Inference

# Environment Setup & Dependency Installation

This cell prepares the environment by:

Suppressing unnecessary warnings

Setting environment variables for stable tokenizer and PyTorch behavior

Removing conflicting package versions

Installing required libraries (Unsloth, datasets, evaluation tools)

In [1]:
import warnings
warnings.filterwarnings("ignore") # Hide unnecessary warnings for cleaner output

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false" # Avoid tokenizer parallelism issues in notebooks
os.environ["PYTORCH_ALLOC_CONF"]     = "expandable_segments:True" # Reduce GPU memory fragmentation

!pip uninstall -y unsloth transformers peft trl accelerate xformers bitsandbytes 2>/dev/null
!pip install --no-cache-dir \
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" \
    pyyaml datasets evaluate rouge_score huggingface_hub -q

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 331.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 189.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 257.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.6/419.6 kB 316.6 MB/

#Config Loader

Loads config.yaml

Validates required sections (model, dataset, training, etc.)

Converts YAML into a Python dictionary

Creates shortcuts (model_cfg, training_cfg, etc.)

In [2]:
import yaml
import pathlib
import json

REQUIRED_SECTIONS = ["model", "lora", "dataset", "training", "inference", "export"]


def load_config_file(config_path: str = "config.yaml") -> dict:

    path = pathlib.Path(config_path)

    if not path.exists():
        raise FileNotFoundError(
            f"\n❌ Config file not found: '{config_path}'\n"
            "   Upload config.yaml to the Colab file panel and re-run."
        )

    with open(path) as file_handle:
        cfg = yaml.safe_load(file_handle)

    missing = [s for s in REQUIRED_SECTIONS if s not in cfg]
    if missing:
        raise KeyError(f"❌ config.yaml is missing sections: {missing}")

    return cfg

cfg = load_config_file("config.yaml")

model_cfg    = cfg["model"]       # unpacked directly → FastLanguageModel.from_pretrained(**model_cfg)
lora_cfg     = cfg["lora"]        # unpacked directly → get_peft_model(model, **lora_cfg)
dataset_cfg  = cfg["dataset"]     # metadata + sources list
training_cfg = cfg["training"]    # unpacked directly → TrainingArguments(**training_cfg)
inference_cfg= cfg["inference"]   # unpacked directly → model.generate(**inference_cfg, …)
export_cfg   = cfg["export"]      # export helpers

enabled_hf = [s["name"] for s in dataset_cfg["sources"]
              if s.get("enabled") and s["type"] == "huggingface"]

print("✅ Config loaded")
print(f"   Model      : {model_cfg['model_name']}")
print(f"   Seq len    : {model_cfg['max_seq_length']}")
print(f"   HF sources : {enabled_hf}")
print(f"   Local file : {dataset_cfg['local_path']}")
print(f"   Cap        : {dataset_cfg['total_cap']}  |  Steps: {training_cfg['max_steps']}")
print(f"   Hub push   : {export_cfg['push_to_hub']} → {export_cfg['hub_model_id']}")


✅ Config loaded
   Model      : unsloth/mistral-7b-bnb-4bit
   Seq len    : 512
   HF sources : ['chatdoctor', 'pubmedqa', 'medmcqa', 'flashcards', 'wikidoc', 'medquad']
   Local file : med_dataset.json
   Cap        : 5000  |  Steps: 500
   Hub push   : True → Akhilesh-2308/Medical-Term_simplifier


# Local Dataset Loader

Loads dataset from a JSON file specified in config

Checks if file exists


Parses JSON into Python list

Prints dataset size and preview

In [3]:
def load_local_dataset(dataset_cfg: dict) -> list:

    local_path = dataset_cfg["local_path"]      # ← from YAML, not hard-coded

    if not pathlib.Path(local_path).exists():
        raise FileNotFoundError(
            f"\n❌ Local dataset not found: '{local_path}'\n"
            "   Upload med_dataset.json via the Colab file panel and retry."
        )

    with open(local_path) as file_handle:
        data = json.load(file_handle)

    print(f"✅ Local dataset: {len(data)} samples  |  keys: {list(data[0].keys())}")
    print(f"   Preview: {data[0]['output'][:120]} …")
    return data


local_data = load_local_dataset(dataset_cfg)

✅ Local dataset: 103 samples  |  keys: ['instruction', 'input', 'output']
   Preview: Hypertension means your blood pressure is consistently higher than normal. Blood pressure measures how hard your blood p …


# Dataset Builder

Loads datasets from multiple sources (local + Hugging Face)

Extracts question-answer pairs from different formats

Normalizes all data into:

{instruction, input, output}

Shuffles, caps, and splits into train/validation

In [5]:
import random
from datasets import load_dataset, Dataset, DatasetDict


def _extract_row(row: dict, src_cfg: dict) -> tuple[str, str]: # Extract question and answer from different dataset formats
    """
    Pull (question, answer) from one raw row of any source.
    All column names come from src_cfg (YAML) — nothing hard-coded.
    Returns ("", "") when data is missing so the caller skips the row.

    Handles three answer layouts:
      1. Direct column           — most sources
      2. Nested dict / list      — PubMedQA (fallback_context_col)
      3. MCQ correct-option idx  — MedMCQA  (correct_idx_col + option_cols)
    """
    question       = str(row.get(src_cfg.get("input_col",  "input"),  "")).strip()
    raw_ans = row.get(src_cfg.get("output_col", "output"))
    answer       = str(raw_ans).strip() if raw_ans and str(raw_ans).lower() not in ("nan","none","") else "" # Case 1: Direct column extraction

    if not answer and src_cfg.get("fallback_context_col"): # Case 2: Handle nested context (e.g., PubMedQA)
        ctx = row.get(src_cfg["fallback_context_col"], {})
        key = src_cfg.get("fallback_context_key", "")
        if isinstance(ctx, dict) and key and key in ctx:
            answer = " ".join(str(p) for p in ctx[key] if p)
        elif isinstance(ctx, list):
            answer = " ".join(str(p) for p in ctx if p)

    if not answer and src_cfg.get("correct_idx_col") and src_cfg.get("option_cols"): # Case 3: Handle MCQ datasets (select correct option)
        idx = row.get(src_cfg["correct_idx_col"])
        if idx is not None:
            try:
                answer = str(row.get(src_cfg["option_cols"][int(idx)], "")).strip()
            except (IndexError, ValueError, TypeError):
                pass

    return question, answer


def load_source(src: dict, variants: list, local_rows: list) -> list: # Load dataset based on source type (local or HuggingFace)
    """
    Load and normalise one data source (local JSON or HuggingFace).
    Returns a list of {instruction, input, output} dicts.
    Does not know about caps, shuffling, or splitting — that's build_dataset's job.
    """
    print(f"\n🔹 Loading '{src['name']}' …")
    limit = src.get("max_samples", 9999)

    if src["type"] == "local":
        rows = local_rows
    elif src["type"] == "huggingface":
        kwargs = {"split": src.get("split", "train")}
        if src.get("config"):
            kwargs["name"] = src["config"]
        ds   = load_dataset(src["path"], **kwargs)
        rows = [dict(r) for r in ds.select(range(min(limit, len(ds))))]
    else:
        raise ValueError(f"Unknown source type: {src['type']}")

    if not rows:
        print("   ⚠️  Empty — skipping")
        return []

    normalised = []
    for row in rows:
        question, answer = _extract_row(row, src)
        if question and answer:
            normalised.append({"instruction": random.choice(variants), "input": question, "output": answer})

    print(f"   ✅ {len(normalised)} usable samples")
    return normalised


def build_dataset(dataset_cfg: dict, local_rows: list) -> DatasetDict:
    """
    Orchestrator: merge all enabled sources → shuffle → cap → train/val split.
    All parameters (seed, cap, val_split, sources) come from dataset_cfg.
    """
    random.seed(dataset_cfg.get("seed", 42))
    all_rows = []

    for src in dataset_cfg["sources"]:
        if not src.get("enabled", True):
            print(f"  ⏭  Skipping '{src['name']}'")
            continue
        try:
            all_rows.extend(load_source(src, dataset_cfg["instruction_variants"], local_rows))
        except Exception as e:
            print(f"  ⚠️  Failed '{src['name']}': {e}")

    random.shuffle(all_rows)
    cap      = dataset_cfg.get("total_cap", 5000) # Limit dataset size to avoid memory issues during training
    val_frac = dataset_cfg.get("val_split",  0.05)
    all_rows = all_rows[:cap]
    cut      = int(len(all_rows) * (1 - val_frac))

    print(f"\n✅ Dataset: {len(all_rows)} samples (cap={cap})")
    return DatasetDict({
        "train":      Dataset.from_list(all_rows[:cut]),
        "validation": Dataset.from_list(all_rows[cut:]),
    })


print("\nBuilding dataset …")
splits = build_dataset(dataset_cfg, local_data)
print(f"   Train: {len(splits['train'])}  |  Val: {len(splits['validation'])}")



Building dataset …

🔹 Loading 'chatdoctor' …
   ✅ 1500 usable samples

🔹 Loading 'pubmedqa' …
   ✅ 700 usable samples

🔹 Loading 'medmcqa' …
   ✅ 700 usable samples

🔹 Loading 'flashcards' …
   ✅ 500 usable samples

🔹 Loading 'wikidoc' …
   ✅ 800 usable samples

🔹 Loading 'medquad' …
   ✅ 800 usable samples

🔹 Loading 'local_curated' …
   ✅ 103 usable samples

✅ Dataset: 5000 samples (cap=5000)
   Train: 4750  |  Val: 250


# Model + LoRA Setup

Loads pretrained model using Unsloth

Applies 4-bit quantization (memory optimization)

Attaches LoRA adapters for efficient fine-tuning

Prints trainable vs total parameters

In [6]:
from unsloth import FastLanguageModel


def load_model(model_cfg: dict, lora_cfg: dict):

    model, tokenizer = FastLanguageModel.from_pretrained(**model_cfg)
    model = FastLanguageModel.get_peft_model(model, **lora_cfg)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"✅ Model + LoRA ready  |  Trainable: {trainable:,}  ({100*trainable/total:.2f}%)")

    return model, tokenizer


model, tokenizer = load_model(model_cfg, lora_cfg)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.5: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.5 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✅ Model + LoRA ready  |  Trainable: 13,631,488  (0.36%)


# Prompt Formatting

Converts dataset into prompt format:

Instruction → Input → Response

Adds EOS token for proper generation stopping

Filters out samples exceeding max sequence length

In [7]:
def build_prompt_formatters(tokenizer, max_seq_length: int):
    eos = tokenizer.eos_token # End-of-sequence token for stopping generation

    def format_row(example: dict) -> dict:
        return {"text": (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Input:\n{example['input']}\n\n"
            f"### Response:\n{example['output']}{eos}"
        )}

    def filter_long(example: dict) -> bool:
        ids = tokenizer(example["text"], truncation=False, add_special_tokens=True)["input_ids"]
        return len(ids) <= max_seq_length

    return format_row, filter_long


def prepare_splits(splits: DatasetDict, tokenizer, model_cfg: dict) -> tuple: # Apply formatting to train and validation datasets

    format_row, filter_long = build_prompt_formatters(tokenizer, model_cfg["max_seq_length"])
    remove_cols = ["instruction", "input", "output"]

    train_ds = splits["train"].map(format_row, remove_columns=remove_cols)
    val_ds   = splits["validation"].map(format_row, remove_columns=remove_cols)

    print("Filtering long sequences …")
    train_ds = train_ds.filter(filter_long, num_proc=1)
    val_ds   = val_ds.filter(filter_long,   num_proc=1)

    print(f"✅ After filter — Train: {len(train_ds)}  |  Val: {len(val_ds)}")
    print(f"   Sample:\n{train_ds[0]['text'][:300]} …")
    return train_ds, val_ds


train_ds, val_ds = prepare_splits(splits, tokenizer, model_cfg)


Map:   0%|          | 0/4750 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Filtering long sequences …


Filter (num_proc=1):   0%|          | 0/4750 [00:00<?, ? examples/s]

Filter (num_proc=1):   0%|          | 0/250 [00:00<?, ? examples/s]

✅ After filter — Train: 4644  |  Val: 244
   Sample:
### Instruction:
What does this medical term mean in everyday language?

### Input:
Do nomograms designed to predict biochemical recurrence (BCR) do a better job of predicting more clinically relevant prostate cancer outcomes than BCR?

### Response:
Currently available nomograms used to predict BCR …


# Trainer Setup + Training

Builds SFTTrainer using model and dataset

Uses config-driven training arguments

Runs training loop

Prints metrics (loss, runtime, speed)

In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

torch._dynamo.config.disable = True


def build_trainer(model, tokenizer, train_ds, val_ds,
                  model_cfg: dict, training_cfg: dict) -> SFTTrainer: # Initialize trainer with model, tokenizer, and datasets
                                                                      # Training arguments are fully controlled via config.yaml
    return SFTTrainer(
        model              = model,
        tokenizer          = tokenizer,
        train_dataset      = train_ds,
        eval_dataset       = val_ds,
        dataset_text_field = "text",
        max_seq_length     = model_cfg["max_seq_length"],
        packing            = False,
        args               = TrainingArguments(**training_cfg),
    )


def run_training(trainer: SFTTrainer) -> dict:
    print("🚀 Starting training …")
    result = trainer.train()
    m = result.metrics
    print(f"\n✅ Training complete")
    print(f"   Runtime     : {m['train_runtime']:.0f}s")
    print(f"   Samples/sec : {m['train_samples_per_second']:.1f}")
    print(f"   Final loss  : {m['train_loss']:.4f}")
    return m


trainer = build_trainer(model, tokenizer, train_ds, val_ds, model_cfg, training_cfg)
metrics = run_training(trainer)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/4644 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/244 [00:00<?, ? examples/s]

🚀 Starting training …


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,644 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 13,631,488 of 7,255,363,584 (0.19% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss
0,1.385089,1.274436



✅ Training complete
   Runtime     : 2876s
   Samples/sec : 1.4
   Final loss  : 1.4327


# Inference

Switches model to evaluation mode

Builds prompt for user input

Generates output using model.generate()

Formats output for readability

Tests multiple medical terms

In [9]:
import textwrap
from typing import Optional
from transformers import logging as hf_logging

hf_logging.set_verbosity_error() # Disable verbose logs for cleaner output
model.eval() # Set model to evaluation mode (no dropout)


def format_output(text: str, width: int = 150) -> str:
    text = text.replace(". ", ".\n")
    return "\n".join(textwrap.fill(line, width=width) for line in text.split("\n"))


def ask(term: str, model, tokenizer,
        inference_cfg: dict, dataset_cfg: dict,
        instruction: Optional[str] = None) -> str:

    if instruction is None:
        instruction = dataset_cfg["instruction_variants"][0]

    prompt = (
        f"### Instruction:\n{instruction}\n\n"
        f"### Input:\n{term}\n\n"
        f"### Response:\n"
    )

    inputs    = tokenizer(prompt, return_tensors="pt", truncation=True).to("cuda")
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            input_ids      = inputs["input_ids"],
            attention_mask = inputs["attention_mask"],
            eos_token_id   = tokenizer.eos_token_id,
            pad_token_id   = tokenizer.eos_token_id,
            use_cache      = True,
            **inference_cfg,

        )

    decoded = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return format_output(decoded)


def run_inference_tests(test_terms: list, model, tokenizer,
                        inference_cfg: dict, dataset_cfg: dict):
    print("\n─── Inference Results ───────────────────────────────────────────")
    for term in test_terms:
        print(f"\n🔹 {term}")
        print(ask(term, model, tokenizer, inference_cfg, dataset_cfg))
    print("─────────────────────────────────────────────────────────────────")


TEST_TERMS = [
    "Bradycardia", "Hyperglycemia", "Sepsis", "Cirrhosis",
    "Pulmonary embolism", "Atrial fibrillation", "Hypothyroidism",
    "Rheumatoid arthritis", "Myocardial infarction", "Hypertensive crisis",
]

run_inference_tests(TEST_TERMS, model, tokenizer, inference_cfg, dataset_cfg)



─── Inference Results ───────────────────────────────────────────

🔹 Bradycardia
Bradycardia is a slow heart rate.
It is defined as fewer than 60 beats per minute (bpm) in adults and fewer than 100 bpm in children.
The normal resting heart rate for an adult is between 60 and 100 bpm.
A slower heart rate may not always be a problem, but it can cause symptoms such as dizziness or fainting if the body does not get enough oxygen.
Bradycardia can be caused by certain medications, low blood pressure, or other conditions that affect the heart's electrical system.
Treatment may involve medication to increase the heart rate or surgery to correct underlying problems with the heart's structure or function.

🔹 Hyperglycemia
Hyperglycemia is a condition where the blood sugar level is higher than normal.
It can be caused by diabetes, stress, certain medications, or other health conditions.
Symptoms may include increased thirst and urination, fatigue, blurred vision, and slow healing of wounds.
Trea

In [10]:
from huggingface_hub import login

from google.colab import userdata

import os

hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token

login(token=hf_token)

model.save_pretrained("outputs/lora_adapter")
tokenizer.save_pretrained("outputs/lora_adapter")

model.push_to_hub("Akhilesh-2308/Medical-Term_simplifier")
tokenizer.push_to_hub("Akhilesh-2308/Medical-Term_simplifier")

README.md:   0%|          | 0.00/553 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|1         |  558kB / 54.6MB            

Saved model to https://huggingface.co/Akhilesh-2308/Medical-Term_simplifier
